###Импорт библиотек и загрузка данных

In [4]:
import pandas as pd
import numpy as np
import altair as alt

alt.renderers.enable('colab')

# 1. Загрузка
DATA_PATH = "person_2025_update.csv.bz2"
df = pd.read_csv(DATA_PATH, low_memory=False, compression='bz2')
# 2. Предобработка
cols = ['hpi', 'birthyear', 'non_en_page_views']
for c in cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

if 'gender' in df.columns:
    df['gender'] = df['gender'].fillna('Unknown').astype(str)

# 3000-5000 точек
source = df.sample(n=min(4000, len(df)), random_state=42).copy()

####Дашборд 1: Анализ стран и профессий

In [8]:
# 1. Селектор для стран
country_select = alt.selection_point(fields=['bplace_country'], name="SelectCountry")

# 2. Левый график: Страны (пульт управления)
top_countries_list = source['bplace_country'].value_counts().head(15).index.tolist()

countries_chart = (
    alt.Chart(source[source['bplace_country'].isin(top_countries_list)])
    .mark_bar()
    .encode(
        y=alt.Y('bplace_country:N', sort='-x', title='Выберите страну'),
        x=alt.X('count():Q', title='Количество людей'),
        color=alt.condition(country_select, alt.value('steelblue'), alt.value('lightgray')),
        tooltip=['bplace_country', 'count()']
    )
    .add_params(country_select)
    .properties(width=200, height=450)
)

# 3. Правый график: Динамический ТОП-10 профессий
occupations_chart = (
    alt.Chart(source)
    .mark_bar()
    .encode(
        x=alt.X('total_count:Q', title='Количество людей'),
        y=alt.Y('occupation:N', sort='-x', title='Топ-10 профессий'),
        color=alt.value('salmon'),
        tooltip=['occupation:N', 'total_count:Q']
    )
    .transform_filter(country_select)
    .transform_aggregate(
        total_count='count()',
        groupby=['occupation']
    )
    .transform_window(
        rank='rank(total_count)',
        sort=[alt.SortField('total_count', order='descending')]
    )
    .transform_filter(alt.datum.rank <= 10)
    .properties(width=500, height=250)
)

# 4. Нижний график: Скаттер (HPI)
scatter_chart = (
    alt.Chart(source)
    .mark_circle(size=30, opacity=0.6, clip=True)
    .encode(
        x=alt.X('birthyear:Q', scale=alt.Scale(domain=[1700, 2000]), title='Год рождения'),
        y=alt.Y('hpi:Q', scale=alt.Scale(domain=[30, 100]), title='HPI'),
        color=alt.Color('gender:N', title="Пол"),
        tooltip=['bplace_country', 'occupation', 'hpi']
    )
    .transform_filter(country_select)
    .properties(width=500, height=200)
)

# Сборка: объединяем по вертикали правые части и приклеиваем левую
(countries_chart | (occupations_chart & scatter_chart)).configure_title(fontSize=16)

alt.HConcatChart(...)

####Дашборд 2: Временная шкала и гендер

In [9]:
# 1. Селекторы
brush = alt.selection_interval(encodings=['x'], name="YearRange")
gender_select = alt.selection_point(fields=['gender'], bind='legend', name="GenderSelect")

# 2. Нижний график: Таймлайн (Кисть)
timeline = (
    alt.Chart(source)
    .mark_area(color='lightgreen', opacity=0.5, line=True)
    .encode(
        x=alt.X('birthyear:Q', bin=alt.Bin(maxbins=50), title='Год рождения'),
        y=alt.Y('count():Q', title='Число рождений')
    )
    .add_params(brush)
    .properties(width=700, height=100, title='Фильтр: Выделите диапазон лет')
)

# 3. Верхний левый: HPI vs Views
scatter_hpi = (
    alt.Chart(source[source['non_en_page_views'] > 0])
    .mark_circle(size=30, clip=True) # clip=True чтобы не вылезало
    .encode(
        x=alt.X('non_en_page_views:Q', scale=alt.Scale(type='log'), title='Просмотры (Log)'),
        y=alt.Y('hpi:Q', scale=alt.Scale(zero=False), title='HPI'),
        color=alt.condition(
            brush,

            alt.Color('gender:N', scale=alt.Scale(scheme='set1'), legend=None),
            alt.value('lightgray')
        ),
        tooltip=['occupation', 'hpi', 'birthyear']
    )
    .add_params(gender_select)
    .transform_filter(gender_select)
    .properties(width=400, height=300, title='HPI vs Популярность')
)

# 4. Верхний правый: Пай-чарт
pie_gender = (
    alt.Chart(source)
    .mark_arc(outerRadius=80)
    .encode(
        theta=alt.Theta('count():Q', stack=True),
        color=alt.Color('gender:N'), # тут тоже лучше явно указать :N
        tooltip=['gender', 'count()'],
        opacity=alt.condition(brush, alt.value(1), alt.value(0.3))
    )
    .transform_filter(brush)
    .transform_filter(gender_select)
    .properties(width=200, height=300, title='Пол (в выбранном диапазоне)')
)

# Сборка
(scatter_hpi | pie_gender) & timeline

alt.VConcatChart(...)